In [13]:
"""
KSH Népszámlálás 2022 – Végleges letöltő és egyesítő script
============================================================

VÉGSŐ MŰKÖDÉS (DEV=False):
  - Mind a 36 dataflow, mind a 3 év (2001, 2011, 2022)
  - Minden fájl külön CSV-be mentve: {ID}_{ÉV}.csv
  - Majd egyesítve két nagy CSV-be:
      ksh_adatok/egyesitett_geo5.csv   – település szintű adatok
      ksh_adatok/egyesitett_geo4.csv   – járás szintű adatok
      ksh_adatok/egyesitett_geo3.csv   – megye szintű adatok
  - Szűrések az egyesítéskor:
      * Csak levél szintű területi kódok (HU, HU11 stb. kiesnek)
      * Csak levél szintű dimenzió értékek (TOTAL kiesik)
      * Területi és dimenzió kódok string-ként tárolva
      * WBL003/WBL018 ALAPSOKASÁG oszloppal megjelölve

DEV=True MŰKÖDÉS:
  - Csak GEO5 dataflow-ok (7 db)
  - Csak 2022
  - Csak az első chunk minden dataflow-nál
  - Fájlnév: {ID}_{ÉV}_dev.csv
  - Egyesítés: ksh_adatok/egyesitett_geo5_dev.csv

pip install requests pandas
"""

import requests
import json
import pandas as pd
import time
import itertools
from pathlib import Path

BASE  = "https://nepszamlalas2022.ksh.hu/api"
VER   = "V67"
LANG  = "hu"
OUT   = Path("ksh_adatok")
OUT.mkdir(exist_ok=True)

# ── Kapcsolók ─────────────────────────────────────────────────
DEV        = True          # True = fejlesztési mód
DEV_YEARS  = ["2022"]
ALL_YEARS  = ["2001", "2011", "2022"]
GEO5_CHUNK = 200           # település / kérés
DELAY      = 1.0
RETRY_DELAY = 30
MAX_RETRIES = 3
COMBO_LIMIT = 50           # max kombinációszám darabolásnál


# ── 36 dataflow definíció ─────────────────────────────────────
# geo:   "GEO5" / "GEO4" / "GEO3"
# large: True = eleve darabolunk, nem várjuk meg a hibát
# alap:  alapsokaság leírása (az egyesített CSV-ben jelöljük)

DATAFLOWS = {
    # GEO5 – Település
    "WBS003": {"name": "Népesség adatai",                               "geo": "GEO5", "large": False, "alap": "teljes_nepesseg"},
    "WBS026": {"name": "Foglalkoztatott népesség",                      "geo": "GEO5", "large": False, "alap": "foglalkoztatottak"},
    "WBS018": {"name": "Háztartástípusok",                              "geo": "GEO5", "large": False, "alap": "haztartasok"},
    "WBH002": {"name": "Háztartások adatai",                           "geo": "GEO5", "large": False, "alap": "haztartasok"},
    "WBC003": {"name": "Családok adatai",                               "geo": "GEO5", "large": False, "alap": "csaladok"},
    "WBL003": {"name": "Lakott lakások jellemzői",                      "geo": "GEO5", "large": False, "alap": "lakott_lakas"},
    "WBL018": {"name": "Lakások legfontosabb jellemzői",                "geo": "GEO5", "large": False, "alap": "osszes_lakas"},
    # GEO4 – Járás
    "WBL001": {"name": "Lakott lakások és lakóik",                      "geo": "GEO4", "large": False, "alap": "lakott_lakas"},
    "WBL004": {"name": "Lakások tulajdonjellege",                       "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL006": {"name": "Lakások helyiségei",                            "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL008": {"name": "Lakások vízellátása",                           "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL010": {"name": "Lakások fűtése",                                "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL016": {"name": "Lakások felszereltsége",                        "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    # GEO3 – Megye
    "WBS004": {"name": "Családi állapot, termékenység",                 "geo": "GEO3", "large": True,  "alap": "15ev_feletti_nepesseg"},
    "WBS005": {"name": "Állampolgárság",                                "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS006": {"name": "Iskolai végzettség",                            "geo": "GEO3", "large": True,  "alap": "15ev_feletti_nepesseg"},
    "WBS007": {"name": "Gazdasági aktivitás",                           "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS008": {"name": "Vallás",                                        "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS009": {"name": "Nemzetiség",                                    "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS010": {"name": "Nyelvismeret",                                  "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS011": {"name": "Digitális jártasság",                           "geo": "GEO3", "large": True,  "alap": "6ev_feletti_nepesseg"},
    "WBS012": {"name": "Egészségi állapot",                             "geo": "GEO3", "large": True,  "alap": "5ev_feletti_nepesseg"},
    "WBS013": {"name": "Iskolába járás",                                "geo": "GEO3", "large": False, "alap": "6ev_feletti_nepesseg"},
    "WBS014": {"name": "Foglalkoztatottak",                             "geo": "GEO3", "large": True,  "alap": "foglalkoztatottak"},
    "WBS015": {"name": "Munkanélküliek",                                "geo": "GEO3", "large": True,  "alap": "munkanelkuliek"},
    "WBS016": {"name": "Foglalkoztatottak nemzetgazdasági ág",          "geo": "GEO3", "large": True,  "alap": "foglalkoztatottak"},
    "WBS017": {"name": "Háztartástípusok",                              "geo": "GEO3", "large": False, "alap": "haztartasok"},
    "WBS019": {"name": "Tanulmányi terület",                            "geo": "GEO3", "large": False, "alap": "kozep_felso_vegzettseg"},
    "WBS021": {"name": "Születéskori lakóhely",                         "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS022": {"name": "Előző lakóhely",                                "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS023": {"name": "Ingázás",                                       "geo": "GEO3", "large": True,  "alap": "foglalkoztatottak"},
    "WBS024": {"name": "Digitális tevékenységek",                       "geo": "GEO3", "large": True,  "alap": "6ev_feletti_nepesseg"},
    "WBS025": {"name": "Korlátozottság",                                "geo": "GEO3", "large": False, "alap": "5ev_feletti_nepesseg"},
    "WBS027": {"name": "Külföldön élt népesség",                        "geo": "GEO3", "large": True,  "alap": "kulfoldot_jart_nepesseg"},
    "WBH001": {"name": "Háztartások adatai",                           "geo": "GEO3", "large": True,  "alap": "haztartasok"},
    "WBC001": {"name": "Családok adatai",                               "geo": "GEO3", "large": False, "alap": "csaladok"},
    "WBC002": {"name": "Családok összetétele",                          "geo": "GEO3", "large": True,  "alap": "csaladok"},
}


# ────────────────────────────────────────────────────────────────
# HTTP
# ────────────────────────────────────────────────────────────────

S = requests.Session()
S.headers.update({"Accept": "application/json",
                   "cache-control": "force-cache"})


def get(path: str) -> dict | list:
    time.sleep(DELAY)
    r = S.get(f"{BASE}{path}", timeout=60)
    r.raise_for_status()
    return r.json()


def fetch_sdmx(sdmx_path: str, timeout: int = 180) -> list[dict] | None:
    url = f"{BASE}/dataflows/{sdmx_path}"
    for attempt in range(1, MAX_RETRIES + 1):
        time.sleep(DELAY)
        raw = b""
        try:
            r = S.get(url, timeout=timeout, stream=True)
            r.raise_for_status()
            for chunk in r.iter_content(65536):
                raw += chunk
        except Exception as e:
            print(f"    ⚠ HTTP hiba ({attempt}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            continue

        text = raw.decode("utf-8", errors="replace").strip()
        if not text.endswith("]"):
            mb = len(raw) / 1024 / 1024
            print(f"    ⚠ Csonka ({mb:.1f} MB, {attempt}/{MAX_RETRIES})")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            continue

        try:
            return json.loads(text)
        except Exception as e:
            print(f"    ⚠ JSON hiba: {e}")
            return None

    return None


# ────────────────────────────────────────────────────────────────
# METAADAT
# ────────────────────────────────────────────────────────────────

def get_structure(dataflow_id: str) -> dict:
    return get(f"/structure/{dataflow_id}/{VER}")


def parse_structure(structure: dict) -> dict:
    dsd = structure["data"]["dataStructures"][0]
    codelists_raw = structure["data"]["codelists"]
    codelists = {}
    for cl in codelists_raw:
        codes = {}
        for code in cl.get("codes", []):
            codes[code["id"]] = {
                "name":    code["names"].get(LANG, code.get("name", "")),
                "name_en": code["names"].get("en", ""),
                "parent":  code.get("parent"),
            }
        codelists[cl["id"]] = {"name": cl["names"].get(LANG, ""), "codes": codes}
    dims = []
    for dim in dsd["dataStructureComponents"]["dimensionList"]["dimensions"]:
        cl_urn = dim["localRepresentation"]["enumeration"]
        cl_id  = cl_urn.split("=")[-1].split("(")[0].split(":")[-1]
        dims.append({
            "position":    dim["position"],
            "id":          dim["id"],
            "codelist_id": cl_id,
            "codelist":    codelists.get(cl_id, {}),
        })
    return {"dimensions": dims, "codelists": codelists}


def get_dim_ids(parsed: dict) -> list[str]:
    return [d["id"] for d in parsed["dimensions"]]


def get_all_codes(parsed: dict, dim_id: str) -> list[str]:
    for dim in parsed["dimensions"]:
        if dim["id"] == dim_id:
            return list(dim["codelist"]["codes"].keys())
    return []


def get_leaf_codes(parsed: dict, dim_id: str) -> list[str]:
    """Kizárja az aggregált kódokat (HU, HU11, TOTAL stb.)"""
    for dim in parsed["dimensions"]:
        if dim["id"] == dim_id:
            codes = dim["codelist"]["codes"]
            parents = {v["parent"] for v in codes.values() if v.get("parent")}
            return [k for k in codes if k not in parents]
    return []


def get_terul_dim(parsed: dict) -> str | None:
    for dim in parsed["dimensions"]:
        if "TERUL_GEO" in dim["id"]:
            return dim["id"]
    return None


def get_split_dims(parsed: dict, n: int = 2) -> list[str]:
    candidates = []
    for dim in parsed["dimensions"]:
        if dim["id"] == "TIME_PERIOD":
            continue
        if "TERUL_" in dim["id"]:
            continue
        n_codes = len(dim["codelist"].get("codes", {}))
        if n_codes > 1:
            candidates.append((n_codes, dim["id"]))
    candidates.sort()
    return [dim_id for _, dim_id in candidates[:n]]


# ────────────────────────────────────────────────────────────────
# SDMX PATH
# ────────────────────────────────────────────────────────────────

def build_path(dataflow_id: str,
               dim_filters: list[tuple[str, list[str] | None]]) -> str:
    parts = []
    for dim_id, values in dim_filters:
        if values:
            parts.append(f"{dim_id}:{'+'.join(values)}")
        else:
            parts.append(dim_id)
    return f"{dataflow_id}/{VER}/d/{','.join(parts)}"


def make_filters(dims: list[str], year: str,
                 pin: dict[str, list[str]] | None = None) -> list[tuple]:
    filters = []
    for d in dims:
        if d == "TIME_PERIOD":
            filters.append((d, [year]))
        elif pin and d in pin:
            filters.append((d, pin[d]))
        else:
            filters.append((d, None))
    return filters


# ────────────────────────────────────────────────────────────────
# SZŰRÉS – TOTAL és aggregált sorok eltávolítása
# ────────────────────────────────────────────────────────────────

def filter_leaf_rows(df: pd.DataFrame, parsed: dict) -> pd.DataFrame:
    """
    Eltávolítja az aggregált sorokat:
    1. Területi aggregátumok (HU, HU11, HU110 stb.) – ezeket már
       a letöltésnél kizárjuk get_leaf_codes-szal, de biztonsági szűrő
    2. Dimenzió TOTAL értékek – ahol a kód maga "TOTAL" vagy
       "_TOTAL"-ra végződik
    """
    mask = pd.Series([True] * len(df), index=df.index)

    for dim in parsed["dimensions"]:
        dim_id = dim["id"]
        if dim_id not in df.columns:
            continue
        if dim_id == "TIME_PERIOD":
            continue

        codes      = dim["codelist"].get("codes", {})
        parents    = {v["parent"] for v in codes.values() if v.get("parent")}
        leaf_set   = {k for k in codes if k not in parents}

        if leaf_set:
            mask &= df[dim_id].astype(str).isin(leaf_set)

    return df[mask].copy()


# ────────────────────────────────────────────────────────────────
# ENRICH + MENTÉS
# ────────────────────────────────────────────────────────────────

def enrich(df: pd.DataFrame, parsed: dict) -> pd.DataFrame:
    if df.empty:
        return df
    dim_to_cl = {d["id"]: d["codelist"] for d in parsed["dimensions"]}
    for col in list(df.columns):
        if col in ("OBS_VALUE", "OBS_STATUS", "DATAFLOW", "DATAFLOW_NEV",
                   "ALAPSOKASÁG", "GEO_SZINT"):
            continue
        codes = dim_to_cl.get(col, {}).get("codes", {})
        if codes:
            df[col + "_nev"] = df[col].astype(str).map(
                lambda x: codes.get(x, {}).get("name", x)
            )
    df["OBS_VALUE"] = pd.to_numeric(df.get("OBS_VALUE"), errors="coerce")
    return df


def save_raw(df: pd.DataFrame, name: str) -> Path:
    """Nyers (szűretlen) fájl mentése – minden letöltött adat."""
    path = OUT / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  💾 {path}  ({len(df):,} sor)")
    return path


# ────────────────────────────────────────────────────────────────
# ADATLEKÉRÉS
# ────────────────────────────────────────────────────────────────

def fetch_geo5(dataflow_id: str, parsed: dict, year: str,
               dev: bool = False) -> pd.DataFrame:
    dims      = get_dim_ids(parsed)
    terul_dim = get_terul_dim(parsed)
    leaves    = get_leaf_codes(parsed, terul_dim)
    n_chunks  = (len(leaves) + GEO5_CHUNK - 1) // GEO5_CHUNK

    print(f"  Levél kódok: {len(leaves)} db, {GEO5_CHUNK}/kérés → {n_chunks} kérés")

    frames = []
    for i in range(0, len(leaves), GEO5_CHUNK):
        chunk     = leaves[i : i + GEO5_CHUNK]
        chunk_num = i // GEO5_CHUNK + 1
        pin       = {terul_dim: chunk}
        sp        = build_path(dataflow_id, make_filters(dims, year, pin))

        print(f"  [{chunk_num:>4}/{n_chunks}] {len(chunk)} telep. "
              f"({chunk[0]}…{chunk[-1]})", end="  ")

        data = fetch_sdmx(sp)
        if data is not None:
            frames.append(pd.DataFrame(data))
            print(f"→ {len(data):,} sor")
        else:
            half = max(1, GEO5_CHUNK // 2)
            print(f"→ felezés ({half})...")
            for j in range(0, len(chunk), half):
                sub     = chunk[j : j + half]
                sub_sp  = build_path(dataflow_id,
                                     make_filters(dims, year, {terul_dim: sub}))
                sub_data = fetch_sdmx(sub_sp)
                if sub_data:
                    frames.append(pd.DataFrame(sub_data))
                    print(f"    ✓ {len(sub_data):,} sor")
                else:
                    print(f"    ✗ {sub[0]}…{sub[-1]}")

        if dev:
            print("  [DEV] csak az első chunk")
            break

    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    # Területi kód string-ként
    if terul_dim in df.columns:
        df[terul_dim] = df[terul_dim].astype(str).str.zfill(5)
    return df


def fetch_geo3_4(dataflow_id: str, parsed: dict, year: str,
                 large: bool = False, dev: bool = False) -> pd.DataFrame:
    dims = get_dim_ids(parsed)

    if not large:
        sp   = build_path(dataflow_id, make_filters(dims, year))
        print(f"  GET /api/dataflows/{sp}")
        data = fetch_sdmx(sp)
        if data is not None:
            print(f"      → {len(data):,} sor")
            df = pd.DataFrame(data)
            return df
        print("  → Váratlanul nagy, darabolásra váltás...")

    split_dims  = get_split_dims(parsed, n=2)
    split_codes = [get_all_codes(parsed, d) for d in split_dims]
    combos      = list(itertools.product(*split_codes))

    if len(combos) > COMBO_LIMIT:
        split_dims  = get_split_dims(parsed, n=3)
        split_codes = [get_all_codes(parsed, d) for d in split_dims]
        combos      = list(itertools.product(*split_codes))

    label_str = " × ".join(
        f"{d}({len(c)})" for d, c in zip(split_dims, split_codes)
    )
    print(f"  → Darabolás: {label_str} = {len(combos)} kérés")

    frames = []
    for idx, combo in enumerate(combos, 1):
        pin   = {dim: [val] for dim, val in zip(split_dims, combo)}
        sp    = build_path(dataflow_id, make_filters(dims, year, pin))
        label = ", ".join(f"{d}={v}" for d, v in zip(split_dims, combo))
        print(f"  [{idx:>4}/{len(combos)}] {label}", end="  ")

        data = fetch_sdmx(sp)
        if data is not None:
            frames.append(pd.DataFrame(data))
            print(f"→ {len(data):,} sor")
        else:
            print("→ ✗ sikertelen")

        if dev:
            print("  [DEV] csak az első kombináció")
            break

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


# ────────────────────────────────────────────────────────────────
# EGYESÍTÉS – egy nagy CSV geo-szintenként
# ────────────────────────────────────────────────────────────────

def build_unified(dataflow_ids: list[str], dev: bool = False) -> pd.DataFrame:
    """
    Beolvassa a letöltött CSV fájlokat, elvégzi a szűréseket
    és egyesíti egy nagy DataFrame-be LONG formátumban.

    Kimenet oszlopai:
      DATAFLOW       - forrás dataflow ID (pl. "WBS003")
      DATAFLOW_NEV   - forrás neve
      ALAPSOKASÁG    - mire vonatkozik (pl. "lakott_lakas")
      GEO_SZINT      - "GEO5" / "GEO4" / "GEO3"
      TIME_PERIOD    - év string ("2001", "2011", "2022")
      TERULET_KOD    - területi kód string, vezető nullákkal ("02112")
      TERULET_NEV    - területi egység neve
      MUTATO_KOD     - mutató kód (pl. "BEG_AR", "ARAM_1")
      MUTATO_NEV     - mutató neve
      OBS_VALUE      - érték (float)
    """
    # Melyik dataflow-ban melyik oszlop a mutató dimenzió
    MUTATO_DIM_MAP = {
        "WBS003": "TEL_SZ_ADAT",
        "WBS026": "FOGL_JEL",
        "WBS018": "HTIP_NEME_KEV1",
        "WBH002": "TEL_H_ADAT",
        "WBC003": "TEL_CS_ADAT",
        "WBL003": "INDICATORS2",
        "WBL018": "INDICATORS3",
    }

    # Egyes dataflow-okban a területi dimenzió neve eltér az alaptól
    TERUL_DIM_OVERRIDE = {
        "WBS026": "TERUL_GEO5_FOGL",  # munkavégzés helye, nem lakóhely
    }

    suffix = "_dev" if dev else ""
    years  = DEV_YEARS if dev else ALL_YEARS
    frames = []

    for fid in dataflow_ids:
        meta = DATAFLOWS[fid]
        for year in years:
            fname = OUT / f"{fid}_{year}{suffix}.csv"
            if not fname.exists():
                print(f"  ⚠ Hiányzó fájl: {fname}")
                continue

            # Minden oszlop string-ként (vezető nullák megmaradnak)
            df = pd.read_csv(fname, dtype=str, low_memory=False)
            if df.empty:
                continue

            # Struktúra a szűréshez
            try:
                s = get_structure(fid)
                p = parse_structure(s)
            except Exception as e:
                print(f"  ⚠ Structure hiba ({fid}): {e}")
                continue

            # Levél szintű szűrés (TOTAL és aggregált kódok kiesnek)
            df = filter_leaf_rows(df, p)
            if df.empty:
                continue

            # Területi kód – vezető nullákkal, string
            terul_dim = TERUL_DIM_OVERRIDE.get(fid) or get_terul_dim(p)
            n_digits  = 5 if meta["geo"] == "GEO5" else (
                        4 if meta["geo"] == "GEO4" else 3)
            terul_kod_col = ""
            terul_nev_col = ""
            if terul_dim and terul_dim in df.columns:
                df[terul_dim] = df[terul_dim].str.strip().str.zfill(n_digits)
                terul_kod_col = terul_dim
                nev_col = terul_dim + "_nev"
                if nev_col in df.columns:
                    terul_nev_col = nev_col

            # Mutató dimenzió
            mutato_dim = MUTATO_DIM_MAP.get(fid)
            if not mutato_dim or mutato_dim not in df.columns:
                skip = {"OBS_STATUS", "OBS_VALUE", "TIME_PERIOD",
                        "TIME_PERIOD_nev", "DATAFLOW", "DATAFLOW_NEV",
                        "ALAPSOKASÁG", "GEO_SZINT"}
                mutato_dim = next(
                    (c for c in df.columns
                     if c not in skip
                     and "TERUL_" not in c
                     and not c.endswith("_nev")
                     and "TIME_" not in c),
                    None
                )
            mutato_nev_col = (mutato_dim + "_nev") if mutato_dim else None

            # Long formátum összerakása
            out_df = pd.DataFrame({
                "DATAFLOW":     fid,
                "DATAFLOW_NEV": meta["name"],
                "ALAPSOKASÁG":  meta["alap"],
                "GEO_SZINT":    meta["geo"],
                "TIME_PERIOD":  df["TIME_PERIOD"].str.strip()
                                if "TIME_PERIOD" in df.columns else year,
                "TERULET_KOD":  df[terul_kod_col] if terul_kod_col else "",
                "TERULET_NEV":  df[terul_nev_col] if terul_nev_col else "",
                "MUTATO_KOD":   df[mutato_dim]
                                if mutato_dim and mutato_dim in df.columns else "",
                "MUTATO_NEV":   df[mutato_nev_col]
                                if mutato_nev_col and mutato_nev_col in df.columns
                                else (df[mutato_dim]
                                      if mutato_dim and mutato_dim in df.columns
                                      else ""),
                "OBS_VALUE":    pd.to_numeric(df["OBS_VALUE"], errors="coerce"),
            })

            # Duplikátumok eltávolítása (előfordulhat több területi dimenzió esetén)
            key_cols = ["TIME_PERIOD", "TERULET_KOD", "MUTATO_KOD"]
            out_df = out_df.drop_duplicates(subset=key_cols)

            # Belső redundáns kódok eltávolítása (azonos tartalom, különböző kód neve)
            # Ezek páronként ugyanazt mérik – a jobb nevű kódot tartjuk meg
            REMOVE_CODES = {
                # WBC003
                "CSG1_1", "CSG1_2", "CSG1_GE3",        # = CSF_1, CSF_2, CSF_GE3
                # WBH002
                "HG1_1", "HG1_2", "HG1_GE3",            # = HF_1, HF_2, HF_GE3
                "HOSSZ21",                               # = H1
                # WBS003
                "FY_15LT",                               # = FY_LT15
                "MA",                                    # = M
                "LT_MWA",                                # = LMS_INV
                # WBL003/WBL018
                "FUTES_TOH_NHEAT21",                     # = FUTES_TOH_NHEAT (részletezés, nem extra info)
                "FUTMODAG_V3_25",                        # = FUTAGOK_FUEL3 (távfűtés)
            }
            out_df = out_df[~out_df["MUTATO_KOD"].isin(REMOVE_CODES)]

            frames.append(out_df)
            print(f"  ✓ {fid} {year}: {len(out_df):,} sor")

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


# ────────────────────────────────────────────────────────────────
# FŐLETÖLTŐ FÜGGVÉNY
# ────────────────────────────────────────────────────────────────

def download(dataflow_ids: list[str], years: list[str],
             dev: bool = False) -> None:
    suffix = "_dev" if dev else ""
    for fid in dataflow_ids:
        meta = DATAFLOWS[fid]
        print(f"\n{'═'*60}")
        print(f"{fid} [{meta['geo']}] – {meta['name']}")

        try:
            s = get_structure(fid)
            p = parse_structure(s)
        except Exception as e:
            print(f"  ⚠ Structure hiba: {e}")
            continue

        for year in years:
            print(f"\n  ── {year} ──")

            out_path = OUT / f"{fid}_{year}{suffix}.csv"
            if not dev and out_path.exists():
                try:
                    existing = pd.read_csv(out_path, low_memory=False,
                                           nrows=1)
                    n = sum(1 for _ in open(out_path)) - 1
                    print(f"  ✓ Már megvan ({n:,} sor)")
                    continue
                except Exception:
                    pass

            try:
                if meta["geo"] == "GEO5":
                    df = fetch_geo5(fid, p, year, dev=dev)
                else:
                    df = fetch_geo3_4(fid, p, year,
                                      large=meta["large"], dev=dev)

                df = enrich(df, p)
                save_raw(df, f"{fid}_{year}{suffix}")

            except Exception as e:
                print(f"  ⚠ Hiba: {e}")

            time.sleep(DELAY)


# ────────────────────────────────────────────────────────────────
# FŐPROGRAM
# ────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    GEO5_IDS = [fid for fid, m in DATAFLOWS.items() if m["geo"] == "GEO5"]
    GEO4_IDS = [fid for fid, m in DATAFLOWS.items() if m["geo"] == "GEO4"]
    GEO3_IDS = [fid for fid, m in DATAFLOWS.items() if m["geo"] == "GEO3"]

    years = DEV_YEARS if DEV else ALL_YEARS

    if DEV:
        # DEV: csak GEO5, csak 2022, csak első chunk
        ids_to_dl = GEO5_IDS
        print("=" * 60)
        print("[DEV MODE] GEO5 dataflow-ok, 2022, első chunk")
        print("=" * 60)
    else:
        # Éles: minden dataflow, minden év
        ids_to_dl = GEO5_IDS + GEO4_IDS + GEO3_IDS
        print("=" * 60)
        print("[ÉLES] Mind a 36 dataflow, 2001 + 2011 + 2022")
        print("=" * 60)

    # 1. Letöltés
    download(ids_to_dl, years, dev=DEV)

    # 2. Egyesítés geo-szintenként
    suffix = "_dev" if DEV else ""
    print(f"\n{'='*60}")
    print("Egyesítés...")

    for geo, ids in [("geo5", GEO5_IDS), ("geo4", GEO4_IDS), ("geo3", GEO3_IDS)]:
        # DEV módban csak GEO5 fut
        if DEV and geo != "geo5":
            continue

        active_ids = [i for i in ids if i in ids_to_dl]
        if not active_ids:
            continue

        print(f"\n── {geo.upper()} ({len(active_ids)} dataflow) ──")
        unified = build_unified(active_ids, dev=DEV)

        if not unified.empty:
            out = OUT / f"egyesitett_{geo}{suffix}.csv"
            unified.to_csv(out, index=False, encoding="utf-8-sig")
            print(f"\n💾 {out}")
            print(f"   {len(unified):,} sor, {len(unified.columns)} oszlop")
            print(f"   Dataflow-ok: {unified['DATAFLOW'].unique()}")
            print(f"   Évek: {sorted(unified['TIME_PERIOD'].dropna().unique())}")
            if "GEO_SZINT" in unified.columns:
                terul_cols = [c for c in unified.columns
                              if "TERUL_GEO" in c and not c.endswith("_nev")]
                if terul_cols:
                    tc = terul_cols[0]
                    print(f"   Területi egységek: {unified[tc].nunique()} db")

    print(f"\n✅ Kész! Fájlok: {OUT}/")

[DEV MODE] GEO5 dataflow-ok, 2022, első chunk

════════════════════════════════════════════════════════════
WBS003 [GEO5] – Népesség adatai

  ── 2022 ──
  Levél kódok: 3178 db, 200/kérés → 16 kérés
  [   1/16] 200 telep. (09566…02370)  → 29,452 sor
  [DEV] csak az első chunk
  💾 ksh_adatok/WBS003_2022_dev.csv  (29,452 sor)

════════════════════════════════════════════════════════════
WBS026 [GEO5] – Foglalkoztatott népesség

  ── 2022 ──
  Levél kódok: 3155 db, 200/kérés → 16 kérés
  [   1/16] 200 telep. (13578…14465)  → 8,000 sor
  [DEV] csak az első chunk
  💾 ksh_adatok/WBS026_2022_dev.csv  (8,000 sor)

════════════════════════════════════════════════════════════
WBS018 [GEO5] – Háztartástípusok

  ── 2022 ──
  Levél kódok: 3178 db, 200/kérés → 16 kérés
  [   1/16] 200 telep. (09566…02370)  → 2,587 sor
  [DEV] csak az első chunk
  💾 ksh_adatok/WBS018_2022_dev.csv  (2,587 sor)

════════════════════════════════════════════════════════════
WBH002 [GEO5] – Háztartások adatai

  ── 2022 

In [1]:
"""
KSH Népszámlálás 2022 – Végleges letöltő és egyesítő script
============================================================

VÉGSŐ MŰKÖDÉS (DEV=False):
  - Mind a 36 dataflow, mind a 3 év (2001, 2011, 2022)
  - Minden fájl külön CSV-be mentve: {ID}_{ÉV}.csv
  - Majd egyesítve két nagy CSV-be:
      ksh_adatok/egyesitett_geo5.csv   – település szintű adatok
      ksh_adatok/egyesitett_geo4.csv   – járás szintű adatok
      ksh_adatok/egyesitett_geo3.csv   – megye szintű adatok
  - Szűrések az egyesítéskor:
      * Csak levél szintű területi kódok (HU, HU11 stb. kiesnek)
      * Csak levél szintű dimenzió értékek (TOTAL kiesik)
      * Területi és dimenzió kódok string-ként tárolva
      * WBL003/WBL018 ALAPSOKASÁG oszloppal megjelölve

DEV=True MŰKÖDÉS:
  - Csak GEO5 dataflow-ok (7 db)
  - Csak 2022
  - Csak az első chunk minden dataflow-nál
  - Fájlnév: {ID}_{ÉV}_dev.csv
  - Egyesítés: ksh_adatok/egyesitett_geo5_dev.csv

pip install requests pandas
"""

import requests
import json
import pandas as pd
import time
import itertools
from pathlib import Path

BASE  = "https://nepszamlalas2022.ksh.hu/api"
VER   = "V67"
LANG  = "hu"
OUT   = Path("ksh_adatok")
OUT.mkdir(exist_ok=True)

# ── Kapcsolók ─────────────────────────────────────────────────
DEV        = False         # True = fejlesztési mód
DEV_YEARS  = ["2022"]
ALL_YEARS  = ["2001", "2011", "2022"]
GEO5_CHUNK = 200           # település / kérés
DELAY      = 1.0
RETRY_DELAY = 30
MAX_RETRIES = 3
COMBO_LIMIT = 50           # max kombinációszám darabolásnál


# ── 36 dataflow definíció ─────────────────────────────────────
# geo:   "GEO5" / "GEO4" / "GEO3"
# large: True = eleve darabolunk, nem várjuk meg a hibát
# alap:  alapsokaság leírása (az egyesített CSV-ben jelöljük)

DATAFLOWS = {
    # GEO5 – Település
    "WBS003": {"name": "Népesség adatai",                               "geo": "GEO5", "large": False, "alap": "teljes_nepesseg"},
    "WBS026": {"name": "Foglalkoztatott népesség",                      "geo": "GEO5", "large": False, "alap": "foglalkoztatottak"},
    "WBS018": {"name": "Háztartástípusok",                              "geo": "GEO5", "large": False, "alap": "haztartasok"},
    "WBH002": {"name": "Háztartások adatai",                           "geo": "GEO5", "large": False, "alap": "haztartasok"},
    "WBC003": {"name": "Családok adatai",                               "geo": "GEO5", "large": False, "alap": "csaladok"},
    "WBL003": {"name": "Lakott lakások jellemzői",                      "geo": "GEO5", "large": False, "alap": "lakott_lakas"},
    "WBL018": {"name": "Lakások legfontosabb jellemzői",                "geo": "GEO5", "large": False, "alap": "osszes_lakas"},
    # GEO4 – Járás
    "WBL001": {"name": "Lakott lakások és lakóik",                      "geo": "GEO4", "large": False, "alap": "lakott_lakas"},
    "WBL004": {"name": "Lakások tulajdonjellege",                       "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL006": {"name": "Lakások helyiségei",                            "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL008": {"name": "Lakások vízellátása",                           "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL010": {"name": "Lakások fűtése",                                "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    "WBL016": {"name": "Lakások felszereltsége",                        "geo": "GEO4", "large": False, "alap": "osszes_lakas"},
    # GEO3 – Megye
    "WBS004": {"name": "Családi állapot, termékenység",                 "geo": "GEO3", "large": True,  "alap": "15ev_feletti_nepesseg"},
    "WBS005": {"name": "Állampolgárság",                                "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS006": {"name": "Iskolai végzettség",                            "geo": "GEO3", "large": True,  "alap": "15ev_feletti_nepesseg"},
    "WBS007": {"name": "Gazdasági aktivitás",                           "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS008": {"name": "Vallás",                                        "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS009": {"name": "Nemzetiség",                                    "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS010": {"name": "Nyelvismeret",                                  "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS011": {"name": "Digitális jártasság",                           "geo": "GEO3", "large": True,  "alap": "6ev_feletti_nepesseg"},
    "WBS012": {"name": "Egészségi állapot",                             "geo": "GEO3", "large": True,  "alap": "5ev_feletti_nepesseg"},
    "WBS013": {"name": "Iskolába járás",                                "geo": "GEO3", "large": False, "alap": "6ev_feletti_nepesseg"},
    "WBS014": {"name": "Foglalkoztatottak",                             "geo": "GEO3", "large": True,  "alap": "foglalkoztatottak"},
    "WBS015": {"name": "Munkanélküliek",                                "geo": "GEO3", "large": True,  "alap": "munkanelkuliek"},
    "WBS016": {"name": "Foglalkoztatottak nemzetgazdasági ág",          "geo": "GEO3", "large": True,  "alap": "foglalkoztatottak"},
    "WBS017": {"name": "Háztartástípusok",                              "geo": "GEO3", "large": False, "alap": "haztartasok"},
    "WBS019": {"name": "Tanulmányi terület",                            "geo": "GEO3", "large": False, "alap": "kozep_felso_vegzettseg"},
    "WBS021": {"name": "Születéskori lakóhely",                         "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS022": {"name": "Előző lakóhely",                                "geo": "GEO3", "large": True,  "alap": "teljes_nepesseg"},
    "WBS023": {"name": "Ingázás",                                       "geo": "GEO3", "large": True,  "alap": "foglalkoztatottak"},
    "WBS024": {"name": "Digitális tevékenységek",                       "geo": "GEO3", "large": True,  "alap": "6ev_feletti_nepesseg"},
    "WBS025": {"name": "Korlátozottság",                                "geo": "GEO3", "large": False, "alap": "5ev_feletti_nepesseg"},
    "WBS027": {"name": "Külföldön élt népesség",                        "geo": "GEO3", "large": True,  "alap": "kulfoldot_jart_nepesseg"},
    "WBH001": {"name": "Háztartások adatai",                           "geo": "GEO3", "large": True,  "alap": "haztartasok"},
    "WBC001": {"name": "Családok adatai",                               "geo": "GEO3", "large": False, "alap": "csaladok"},
    "WBC002": {"name": "Családok összetétele",                          "geo": "GEO3", "large": True,  "alap": "csaladok"},
}


# ────────────────────────────────────────────────────────────────
# HTTP
# ────────────────────────────────────────────────────────────────

S = requests.Session()
S.headers.update({"Accept": "application/json",
                   "cache-control": "force-cache"})


def get(path: str) -> dict | list:
    time.sleep(DELAY)
    r = S.get(f"{BASE}{path}", timeout=60)
    r.raise_for_status()
    return r.json()


def fetch_sdmx(sdmx_path: str, timeout: int = 180) -> list[dict] | None:
    url = f"{BASE}/dataflows/{sdmx_path}"
    for attempt in range(1, MAX_RETRIES + 1):
        time.sleep(DELAY)
        raw = b""
        try:
            r = S.get(url, timeout=timeout, stream=True)
            r.raise_for_status()
            for chunk in r.iter_content(65536):
                raw += chunk
        except Exception as e:
            print(f"    ⚠ HTTP hiba ({attempt}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            continue

        text = raw.decode("utf-8", errors="replace").strip()
        if not text.endswith("]"):
            mb = len(raw) / 1024 / 1024
            print(f"    ⚠ Csonka ({mb:.1f} MB, {attempt}/{MAX_RETRIES})")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            continue

        try:
            return json.loads(text)
        except Exception as e:
            print(f"    ⚠ JSON hiba: {e}")
            return None

    return None


# ────────────────────────────────────────────────────────────────
# METAADAT
# ────────────────────────────────────────────────────────────────

def get_structure(dataflow_id: str) -> dict:
    return get(f"/structure/{dataflow_id}/{VER}")


def parse_structure(structure: dict) -> dict:
    dsd = structure["data"]["dataStructures"][0]
    codelists_raw = structure["data"]["codelists"]
    codelists = {}
    for cl in codelists_raw:
        codes = {}
        for code in cl.get("codes", []):
            codes[code["id"]] = {
                "name":    code["names"].get(LANG, code.get("name", "")),
                "name_en": code["names"].get("en", ""),
                "parent":  code.get("parent"),
            }
        codelists[cl["id"]] = {"name": cl["names"].get(LANG, ""), "codes": codes}
    dims = []
    for dim in dsd["dataStructureComponents"]["dimensionList"]["dimensions"]:
        cl_urn = dim["localRepresentation"]["enumeration"]
        cl_id  = cl_urn.split("=")[-1].split("(")[0].split(":")[-1]
        dims.append({
            "position":    dim["position"],
            "id":          dim["id"],
            "codelist_id": cl_id,
            "codelist":    codelists.get(cl_id, {}),
        })
    return {"dimensions": dims, "codelists": codelists}


def get_dim_ids(parsed: dict) -> list[str]:
    return [d["id"] for d in parsed["dimensions"]]


def get_all_codes(parsed: dict, dim_id: str) -> list[str]:
    for dim in parsed["dimensions"]:
        if dim["id"] == dim_id:
            return list(dim["codelist"]["codes"].keys())
    return []


def get_leaf_codes(parsed: dict, dim_id: str) -> list[str]:
    """Kizárja az aggregált kódokat (HU, HU11, TOTAL stb.)"""
    for dim in parsed["dimensions"]:
        if dim["id"] == dim_id:
            codes = dim["codelist"]["codes"]
            parents = {v["parent"] for v in codes.values() if v.get("parent")}
            return [k for k in codes if k not in parents]
    return []


def get_terul_dim(parsed: dict) -> str | None:
    for dim in parsed["dimensions"]:
        if "TERUL_GEO" in dim["id"]:
            return dim["id"]
    return None


def get_split_dims(parsed: dict, n: int = 2) -> list[str]:
    candidates = []
    for dim in parsed["dimensions"]:
        if dim["id"] == "TIME_PERIOD":
            continue
        if "TERUL_" in dim["id"]:
            continue
        n_codes = len(dim["codelist"].get("codes", {}))
        if n_codes > 1:
            candidates.append((n_codes, dim["id"]))
    candidates.sort()
    return [dim_id for _, dim_id in candidates[:n]]


# ────────────────────────────────────────────────────────────────
# SDMX PATH
# ────────────────────────────────────────────────────────────────

def build_path(dataflow_id: str,
               dim_filters: list[tuple[str, list[str] | None]]) -> str:
    parts = []
    for dim_id, values in dim_filters:
        if values:
            parts.append(f"{dim_id}:{'+'.join(values)}")
        else:
            parts.append(dim_id)
    return f"{dataflow_id}/{VER}/d/{','.join(parts)}"


def make_filters(dims: list[str], year: str,
                 pin: dict[str, list[str]] | None = None) -> list[tuple]:
    filters = []
    for d in dims:
        if d == "TIME_PERIOD":
            filters.append((d, [year]))
        elif pin and d in pin:
            filters.append((d, pin[d]))
        else:
            filters.append((d, None))
    return filters


# ────────────────────────────────────────────────────────────────
# SZŰRÉS – TOTAL és aggregált sorok eltávolítása
# ────────────────────────────────────────────────────────────────

def filter_leaf_rows(df: pd.DataFrame, parsed: dict) -> pd.DataFrame:
    """
    Eltávolítja az aggregált sorokat:
    1. Területi aggregátumok (HU, HU11, HU110 stb.) – ezeket már
       a letöltésnél kizárjuk get_leaf_codes-szal, de biztonsági szűrő
    2. Dimenzió TOTAL értékek – ahol a kód maga "TOTAL" vagy
       "_TOTAL"-ra végződik
    """
    mask = pd.Series([True] * len(df), index=df.index)

    for dim in parsed["dimensions"]:
        dim_id = dim["id"]
        if dim_id not in df.columns:
            continue
        if dim_id == "TIME_PERIOD":
            continue

        codes      = dim["codelist"].get("codes", {})
        parents    = {v["parent"] for v in codes.values() if v.get("parent")}
        leaf_set   = {k for k in codes if k not in parents}

        if leaf_set:
            mask &= df[dim_id].astype(str).isin(leaf_set)

    return df[mask].copy()


# ────────────────────────────────────────────────────────────────
# ENRICH + MENTÉS
# ────────────────────────────────────────────────────────────────

def enrich(df: pd.DataFrame, parsed: dict) -> pd.DataFrame:
    if df.empty:
        return df
    dim_to_cl = {d["id"]: d["codelist"] for d in parsed["dimensions"]}
    for col in list(df.columns):
        if col in ("OBS_VALUE", "OBS_STATUS", "DATAFLOW", "DATAFLOW_NEV",
                   "ALAPSOKASÁG", "GEO_SZINT"):
            continue
        codes = dim_to_cl.get(col, {}).get("codes", {})
        if codes:
            df[col + "_nev"] = df[col].astype(str).map(
                lambda x: codes.get(x, {}).get("name", x)
            )
    df["OBS_VALUE"] = pd.to_numeric(df.get("OBS_VALUE"), errors="coerce")
    return df


def save_raw(df: pd.DataFrame, name: str) -> Path:
    """Nyers (szűretlen) fájl mentése – minden letöltött adat."""
    path = OUT / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  💾 {path}  ({len(df):,} sor)")
    return path


# ────────────────────────────────────────────────────────────────
# ADATLEKÉRÉS
# ────────────────────────────────────────────────────────────────

def fetch_geo5(dataflow_id: str, parsed: dict, year: str,
               dev: bool = False) -> pd.DataFrame:
    dims      = get_dim_ids(parsed)
    terul_dim = get_terul_dim(parsed)
    leaves    = get_leaf_codes(parsed, terul_dim)
    n_chunks  = (len(leaves) + GEO5_CHUNK - 1) // GEO5_CHUNK

    print(f"  Levél kódok: {len(leaves)} db, {GEO5_CHUNK}/kérés → {n_chunks} kérés")

    frames = []
    for i in range(0, len(leaves), GEO5_CHUNK):
        chunk     = leaves[i : i + GEO5_CHUNK]
        chunk_num = i // GEO5_CHUNK + 1
        pin       = {terul_dim: chunk}
        sp        = build_path(dataflow_id, make_filters(dims, year, pin))

        print(f"  [{chunk_num:>4}/{n_chunks}] {len(chunk)} telep. "
              f"({chunk[0]}…{chunk[-1]})", end="  ")

        data = fetch_sdmx(sp)
        if data is not None:
            frames.append(pd.DataFrame(data))
            print(f"→ {len(data):,} sor")
        else:
            half = max(1, GEO5_CHUNK // 2)
            print(f"→ felezés ({half})...")
            for j in range(0, len(chunk), half):
                sub     = chunk[j : j + half]
                sub_sp  = build_path(dataflow_id,
                                     make_filters(dims, year, {terul_dim: sub}))
                sub_data = fetch_sdmx(sub_sp)
                if sub_data:
                    frames.append(pd.DataFrame(sub_data))
                    print(f"    ✓ {len(sub_data):,} sor")
                else:
                    print(f"    ✗ {sub[0]}…{sub[-1]}")

        if dev:
            print("  [DEV] csak az első chunk")
            break

    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    # Területi kód string-ként
    if terul_dim in df.columns:
        df[terul_dim] = df[terul_dim].astype(str).str.zfill(5)
    return df


def fetch_geo3_4(dataflow_id: str, parsed: dict, year: str,
                 large: bool = False, dev: bool = False) -> pd.DataFrame:
    dims = get_dim_ids(parsed)

    if not large:
        sp   = build_path(dataflow_id, make_filters(dims, year))
        print(f"  GET /api/dataflows/{sp}")
        data = fetch_sdmx(sp)
        if data is not None:
            print(f"      → {len(data):,} sor")
            df = pd.DataFrame(data)
            return df
        print("  → Váratlanul nagy, darabolásra váltás...")

    split_dims  = get_split_dims(parsed, n=2)
    split_codes = [get_all_codes(parsed, d) for d in split_dims]
    combos      = list(itertools.product(*split_codes))

    if len(combos) > COMBO_LIMIT:
        split_dims  = get_split_dims(parsed, n=3)
        split_codes = [get_all_codes(parsed, d) for d in split_dims]
        combos      = list(itertools.product(*split_codes))

    label_str = " × ".join(
        f"{d}({len(c)})" for d, c in zip(split_dims, split_codes)
    )
    print(f"  → Darabolás: {label_str} = {len(combos)} kérés")

    frames = []
    for idx, combo in enumerate(combos, 1):
        pin   = {dim: [val] for dim, val in zip(split_dims, combo)}
        sp    = build_path(dataflow_id, make_filters(dims, year, pin))
        label = ", ".join(f"{d}={v}" for d, v in zip(split_dims, combo))
        print(f"  [{idx:>4}/{len(combos)}] {label}", end="  ")

        data = fetch_sdmx(sp)
        if data is not None:
            frames.append(pd.DataFrame(data))
            print(f"→ {len(data):,} sor")
        else:
            print("→ ✗ sikertelen")

        if dev:
            print("  [DEV] csak az első kombináció")
            break

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


# ────────────────────────────────────────────────────────────────
# EGYESÍTÉS – egy nagy CSV geo-szintenként
# ────────────────────────────────────────────────────────────────

def build_unified(dataflow_ids: list[str], dev: bool = False) -> pd.DataFrame:
    """
    Beolvassa a letöltött CSV fájlokat, elvégzi a szűréseket
    és egyesíti egy nagy DataFrame-be LONG formátumban.

    Kimenet oszlopai:
      DATAFLOW       - forrás dataflow ID (pl. "WBS003")
      DATAFLOW_NEV   - forrás neve
      ALAPSOKASÁG    - mire vonatkozik (pl. "lakott_lakas")
      GEO_SZINT      - "GEO5" / "GEO4" / "GEO3"
      TIME_PERIOD    - év string ("2001", "2011", "2022")
      TERULET_KOD    - területi kód string, vezető nullákkal ("02112")
      TERULET_NEV    - területi egység neve
      MUTATO_KOD     - mutató kód (pl. "BEG_AR", "ARAM_1")
      MUTATO_NEV     - mutató neve
      OBS_VALUE      - érték (float)
    """
    # Melyik dataflow-ban melyik oszlop a mutató dimenzió
    MUTATO_DIM_MAP = {
        "WBS003": "TEL_SZ_ADAT",
        "WBS026": "FOGL_JEL",
        "WBS018": "HTIP_NEME_KEV1",
        "WBH002": "TEL_H_ADAT",
        "WBC003": "TEL_CS_ADAT",
        "WBL003": "INDICATORS2",
        "WBL018": "INDICATORS3",
    }

    # Egyes dataflow-okban a területi dimenzió neve eltér az alaptól
    TERUL_DIM_OVERRIDE = {
        "WBS026": "TERUL_GEO5_FOGL",  # munkavégzés helye, nem lakóhely
    }

    suffix = "_dev" if dev else ""
    years  = DEV_YEARS if dev else ALL_YEARS
    frames = []

    for fid in dataflow_ids:
        meta = DATAFLOWS[fid]
        for year in years:
            fname = OUT / f"{fid}_{year}{suffix}.csv"
            if not fname.exists():
                print(f"  ⚠ Hiányzó fájl: {fname}")
                continue

            # Minden oszlop string-ként (vezető nullák megmaradnak)
            df = pd.read_csv(fname, dtype=str, low_memory=False)
            if df.empty:
                continue

            # Struktúra a szűréshez
            try:
                s = get_structure(fid)
                p = parse_structure(s)
            except Exception as e:
                print(f"  ⚠ Structure hiba ({fid}): {e}")
                continue

            # Levél szintű szűrés (TOTAL és aggregált kódok kiesnek)
            df = filter_leaf_rows(df, p)
            if df.empty:
                continue

            # Területi kód – vezető nullákkal, string
            terul_dim = TERUL_DIM_OVERRIDE.get(fid) or get_terul_dim(p)
            n_digits  = 5 if meta["geo"] == "GEO5" else (
                        4 if meta["geo"] == "GEO4" else 3)
            terul_kod_col = ""
            terul_nev_col = ""
            if terul_dim and terul_dim in df.columns:
                df[terul_dim] = df[terul_dim].str.strip().str.zfill(n_digits)
                terul_kod_col = terul_dim
                nev_col = terul_dim + "_nev"
                if nev_col in df.columns:
                    terul_nev_col = nev_col

            # Mutató dimenzió
            mutato_dim = MUTATO_DIM_MAP.get(fid)
            if not mutato_dim or mutato_dim not in df.columns:
                skip = {"OBS_STATUS", "OBS_VALUE", "TIME_PERIOD",
                        "TIME_PERIOD_nev", "DATAFLOW", "DATAFLOW_NEV",
                        "ALAPSOKASÁG", "GEO_SZINT"}
                mutato_dim = next(
                    (c for c in df.columns
                     if c not in skip
                     and "TERUL_" not in c
                     and not c.endswith("_nev")
                     and "TIME_" not in c),
                    None
                )
            mutato_nev_col = (mutato_dim + "_nev") if mutato_dim else None

            # Long formátum összerakása
            out_df = pd.DataFrame({
                "DATAFLOW":     fid,
                "DATAFLOW_NEV": meta["name"],
                "ALAPSOKASÁG":  meta["alap"],
                "GEO_SZINT":    meta["geo"],
                "TIME_PERIOD":  df["TIME_PERIOD"].str.strip()
                                if "TIME_PERIOD" in df.columns else year,
                "TERULET_KOD":  df[terul_kod_col] if terul_kod_col else "",
                "TERULET_NEV":  df[terul_nev_col] if terul_nev_col else "",
                "MUTATO_KOD":   df[mutato_dim]
                                if mutato_dim and mutato_dim in df.columns else "",
                "MUTATO_NEV":   df[mutato_nev_col]
                                if mutato_nev_col and mutato_nev_col in df.columns
                                else (df[mutato_dim]
                                      if mutato_dim and mutato_dim in df.columns
                                      else ""),
                "OBS_VALUE":    pd.to_numeric(df["OBS_VALUE"], errors="coerce"),
            })

            # Duplikátumok eltávolítása (előfordulhat több területi dimenzió esetén)
            key_cols = ["TIME_PERIOD", "TERULET_KOD", "MUTATO_KOD"]
            out_df = out_df.drop_duplicates(subset=key_cols)

            # Belső redundáns kódok eltávolítása (azonos tartalom, különböző kód neve)
            # Ezek páronként ugyanazt mérik – a jobb nevű kódot tartjuk meg
            REMOVE_CODES = {
                # WBC003
                "CSG1_1", "CSG1_2", "CSG1_GE3",        # = CSF_1, CSF_2, CSF_GE3
                # WBH002
                "HG1_1", "HG1_2", "HG1_GE3",            # = HF_1, HF_2, HF_GE3
                "HOSSZ21",                               # = H1
                # WBS003
                "FY_15LT",                               # = FY_LT15
                "MA",                                    # = M
                "LT_MWA",                                # = LMS_INV
                # WBL003/WBL018
                "FUTES_TOH_NHEAT21",                     # = FUTES_TOH_NHEAT (részletezés, nem extra info)
                "FUTMODAG_V3_25",                        # = FUTAGOK_FUEL3 (távfűtés)
            }
            out_df = out_df[~out_df["MUTATO_KOD"].isin(REMOVE_CODES)]

            frames.append(out_df)
            print(f"  ✓ {fid} {year}: {len(out_df):,} sor")

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


# ────────────────────────────────────────────────────────────────
# FŐLETÖLTŐ FÜGGVÉNY
# ────────────────────────────────────────────────────────────────

def download(dataflow_ids: list[str], years: list[str],
             dev: bool = False) -> None:
    suffix = "_dev" if dev else ""
    for fid in dataflow_ids:
        meta = DATAFLOWS[fid]
        print(f"\n{'═'*60}")
        print(f"{fid} [{meta['geo']}] – {meta['name']}")

        try:
            s = get_structure(fid)
            p = parse_structure(s)
        except Exception as e:
            print(f"  ⚠ Structure hiba: {e}")
            continue

        for year in years:
            print(f"\n  ── {year} ──")

            out_path = OUT / f"{fid}_{year}{suffix}.csv"
            if not dev and out_path.exists():
                try:
                    existing = pd.read_csv(out_path, low_memory=False,
                                           nrows=1)
                    n = sum(1 for _ in open(out_path)) - 1
                    print(f"  ✓ Már megvan ({n:,} sor)")
                    continue
                except Exception:
                    pass

            try:
                if meta["geo"] == "GEO5":
                    df = fetch_geo5(fid, p, year, dev=dev)
                else:
                    df = fetch_geo3_4(fid, p, year,
                                      large=meta["large"], dev=dev)

                df = enrich(df, p)
                save_raw(df, f"{fid}_{year}{suffix}")

            except Exception as e:
                print(f"  ⚠ Hiba: {e}")

            time.sleep(DELAY)


# ────────────────────────────────────────────────────────────────
# FŐPROGRAM
# ────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    GEO5_IDS = [fid for fid, m in DATAFLOWS.items() if m["geo"] == "GEO5"]
    GEO4_IDS = [fid for fid, m in DATAFLOWS.items() if m["geo"] == "GEO4"]
    GEO3_IDS = [fid for fid, m in DATAFLOWS.items() if m["geo"] == "GEO3"]

    years = DEV_YEARS if DEV else ALL_YEARS

    if DEV:
        # DEV: csak GEO5, csak 2022, csak első chunk
        ids_to_dl = GEO5_IDS
        print("=" * 60)
        print("[DEV MODE] GEO5 dataflow-ok, 2022, első chunk")
        print("=" * 60)
    else:
        # Éles: minden dataflow, minden év
        ids_to_dl = GEO5_IDS  # TODO: GEO4_IDS + GEO3_IDS hozzáadása ha készen áll
        print("=" * 60)
        print("[ÉLES] Mind a 36 dataflow, 2001 + 2011 + 2022")
        print("=" * 60)

    # 1. Letöltés
    download(ids_to_dl, years, dev=DEV)

    # 2. Egyesítés geo-szintenként
    suffix = "_dev" if DEV else ""
    print(f"\n{'='*60}")
    print("Egyesítés...")

    for geo, ids in [("geo5", GEO5_IDS), ("geo4", GEO4_IDS), ("geo3", GEO3_IDS)]:
        # DEV módban csak GEO5 fut
        if DEV and geo != "geo5":
            continue

        active_ids = [i for i in ids if i in ids_to_dl]
        if not active_ids:
            continue

        print(f"\n── {geo.upper()} ({len(active_ids)} dataflow) ──")
        unified = build_unified(active_ids, dev=DEV)

        if not unified.empty:
            out = OUT / f"egyesitett_{geo}{suffix}.csv"
            unified.to_csv(out, index=False, encoding="utf-8-sig")
            print(f"\n💾 {out}")
            print(f"   {len(unified):,} sor, {len(unified.columns)} oszlop")
            print(f"   Dataflow-ok: {unified['DATAFLOW'].unique()}")
            print(f"   Évek: {sorted(unified['TIME_PERIOD'].dropna().unique())}")
            if "GEO_SZINT" in unified.columns:
                terul_cols = [c for c in unified.columns
                              if "TERUL_GEO" in c and not c.endswith("_nev")]
                if terul_cols:
                    tc = terul_cols[0]
                    print(f"   Területi egységek: {unified[tc].nunique()} db")

    print(f"\n✅ Kész! Fájlok: {OUT}/")

[ÉLES] Mind a 36 dataflow, 2001 + 2011 + 2022

════════════════════════════════════════════════════════════
WBS003 [GEO5] – Népesség adatai

  ── 2001 ──
  ✓ Már megvan (438,564 sor)

  ── 2011 ──
  ✓ Már megvan (438,426 sor)

  ── 2022 ──
  ✓ Már megvan (470,196 sor)

════════════════════════════════════════════════════════════
WBS026 [GEO5] – Foglalkoztatott népesség

  ── 2001 ──
  ✓ Már megvan (126,200 sor)

  ── 2011 ──
  ✓ Már megvan (126,200 sor)

  ── 2022 ──
  ✓ Már megvan (126,200 sor)

════════════════════════════════════════════════════════════
WBS018 [GEO5] – Háztartástípusok

  ── 2001 ──
  ✓ Már megvan (41,314 sor)

  ── 2011 ──
  ✓ Már megvan (41,301 sor)

  ── 2022 ──
  ✓ Már megvan (41,301 sor)

════════════════════════════════════════════════════════════
WBH002 [GEO5] – Háztartások adatai

  ── 2001 ──
  ✓ Már megvan (276,399 sor)

  ── 2011 ──
  ✓ Már megvan (276,399 sor)

  ── 2022 ──
  ✓ Már megvan (276,399 sor)

═══════════════════════════════════════════════════

In [3]:
import pandas as pd
df = pd.read_csv("ksh_adatok/egyesitett_geo5.csv", dtype=str, low_memory=False)

# Vezető nullák megvannak-e?
print("TERULET_KOD első 10 egyedi érték (rendezve):")
print(sorted(df["TERULET_KOD"].dropna().unique())[:10])

# Budapest kerületek (001-023) megvannak-e?
bp = [k for k in df["TERULET_KOD"].unique() if len(str(k)) <= 3]
print(f"\n3 vagy kevesebb karakterű kódok: {sorted(bp)[:10]}")

TERULET_KOD első 10 egyedi érték (rendezve):
['01508', '01526', '02015', '02024', '02033', '02042', '02051', '02060', '02079', '02088']

3 vagy kevesebb karakterű kódok: []
